In [1]:
# ------------------------------
# 1️⃣ Create Spark Session with Delta Lake
# ------------------------------
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Mailchimp Delta Example") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")  # optional: reduce log spam

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a3598ad2-f63f-4f7b-9e6d-816d4018aa5d;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 303ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |

In [2]:
# ------------------------------
# 2️⃣ Read CSV file
# ------------------------------
csv_path = "/home/jovyan/work/joined_contacts_campaigns.csv"  # exact path in Jupyter container

df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
print("Schema of CSV:")
df.printSchema()
df.show(5)


Schema of CSV:
root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- phone: double (nullable = true)
 |-- email: string (nullable = true)
 |-- company: string (nullable = true)
 |-- lead_source: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- deal_value: double (nullable = true)
 |-- clv: double (nullable = true)
 |-- created_date: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- send_date: date (nullable = true)
 |-- audience_size: integer (nullable = true)
 |-- emails_sent: integer (nullable = true)
 |-- opens: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- conversions: integer (nullable = true)
 |-- spend: double (nullable = true)
 |-- revenue: double (nullable = true)

+----------+---------+----------+--------------------+--------------------+-----------+---------------+----------+-----

In [3]:
# ------------------------------
# 3️⃣ Convert to Delta Lake
# ------------------------------
delta_path = "/home/jovyan/work/delta_lake"  # folder to store delta table

df.write.format("delta").mode("overwrite").save(delta_path)

# ------------------------------
# 4️⃣ Read Delta Table to verify
# ------------------------------
df_delta = spark.read.format("delta").load(delta_path)
print("Delta Table Preview:")
df_delta.show(5)

25/09/22 13:42:41 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta Table Preview:


+----------+---------+----------+--------------------+--------------------+-----------+---------------+----------+--------+------------+-----------+-----------------+--------+----------+-------------+-----------+-----+------+-----------+-------+-------+
|first_name|last_name|     phone|               email|             company|lead_source|lifecycle_stage|deal_value|     clv|created_date|campaign_id|    campaign_name| channel| send_date|audience_size|emails_sent|opens|clicks|conversions|  spend|revenue|
+----------+---------+----------+--------------------+--------------------+-----------+---------------+----------+--------+------------+-----------+-----------------+--------+----------+-------------+-----------+-----+------+-----------+-------+-------+
|     Kayla|  Johnson|6.71773E14|kayla.johnson45@g...|Duarte, Smith and...|   LinkedIn|     subscriber|   7485.72| 37428.6|  19-01-2024|   CMP-2240|      Spring Sale|   Email|2025-09-17|         4676|       4037| 1282|   217|         86| 